## DDL Gold: pf.gold.dim_modelo_scd2  (SCD Type 2)
## Guarda historial de atributos del modelo.
## Regla: cerrar version vigente, luego insertar la nueva.

In [0]:
%sql
DROP TABLE IF EXISTS pf.gold.dim_modelo_scd2;

CREATE TABLE IF NOT EXISTS pf.gold.dim_modelo_scd2 (
    modelo_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'PK - Surrogate Key',
    model_id STRING NOT NULL COMMENT 'BK - ID natural org/nombre',
    nombre STRING COMMENT 'Nombre corto',
    org_id STRING COMMENT 'Organizacion',
    pipeline_tag STRING COMMENT 'Tarea',
    library_name STRING COMMENT 'Libreria',
    license_tag STRING COMMENT 'Licencia (license:*)',
    valid_from TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'Inicio de vigencia',
    valid_to TIMESTAMP DEFAULT '9999-12-31' COMMENT 'Fin de vigencia',
    is_current BOOLEAN DEFAULT TRUE COMMENT 'True solo para la version vigente',
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'UTC',
    PRIMARY KEY (modelo_id)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Dimension Modelo - SCD Type 2 (historico de atributos)';

In [0]:
%sql
-- =========================================================
-- POPULADO SCD2 (2 operaciones: cerrar + insertar)
-- Staging: el snapshot hoy de Silver
-- =========================================================
CREATE OR REPLACE TEMP VIEW staging_modelos AS
SELECT model_id, nombre, org_id, pipeline_tag, library_name, COALESCE(license_tag, 'sin_licencia') AS license_tag
FROM pf.silver.modelos
WHERE ingestion_date = (SELECT MAX(ingestion_date) FROM pf.silver.modelos);

-- PASO 1: cerrar version vigente si algun atributo cambio
MERGE INTO pf.gold.dim_modelo_scd2 AS t
USING staging_modelos AS s
ON  t.model_id = s.model_id AND t.is_current = TRUE
WHEN MATCHED AND (
        t.nombre        IS DISTINCT FROM s.nombre OR
        t.org_id        IS DISTINCT FROM s.org_id OR
        t.pipeline_tag  IS DISTINCT FROM s.pipeline_tag OR
        t.library_name  IS DISTINCT FROM s.library_name OR
        t.license_tag   IS DISTINCT FROM s.license_tag
     )
THEN UPDATE SET t.valid_to = CURRENT_TIMESTAMP(), t.is_current = FALSE;

-- PASO 2: insertar nueva version vigente (si no existe)
INSERT INTO pf.gold.dim_modelo_scd2 (model_id, nombre, org_id, pipeline_tag, library_name, license_tag, valid_from, valid_to, is_current, _createdAt)
SELECT s.model_id, s.nombre, s.org_id, s.pipeline_tag, s.library_name, s.license_tag,
       CURRENT_TIMESTAMP(), '9999-12-31', TRUE, CURRENT_TIMESTAMP()
FROM staging_modelos s
WHERE NOT EXISTS (
    SELECT 1 FROM pf.gold.dim_modelo_scd2 t
    WHERE t.model_id = s.model_id AND t.is_current = TRUE
);

In [0]:
%sql

-- Verificacion: versiones vigentes y conteo de historico
SELECT is_current, COUNT(*) AS n
FROM pf.gold.dim_modelo_scd2
GROUP BY is_current;

SELECT modelo_id, model_id, nombre, pipeline_tag, license_tag, is_current
FROM pf.gold.dim_modelo_scd2
WHERE is_current = TRUE
ORDER BY model_id
LIMIT 10;